In [5]:
from notebooks.features.feature_extraction import load_all_features

loaded_features = load_all_features(version='v2')

In [6]:
import pandas as pd

from notebooks.consts import PROCESSED_CSV, NOTEBOOK_PATH

data = pd.read_csv(PROCESSED_CSV)
final_data = pd.merge(loaded_features, data, on='index_v2')

In [7]:
from tauso.data.consts import CANONICAL_GENE
target_linkages = ['phosphorothioate', 'phosphorothioate/phosphodiester']

final_data = final_data[final_data[CANONICAL_GENE] != 'HBV']
final_data = final_data[final_data['Linkage'].isin(target_linkages)].copy()

In [8]:
import os
import xgboost as xgb
import numpy as np

def predict_aso_efficacy(model_path: str, input_df: pd.DataFrame) -> np.ndarray:
    """
    Loads a saved XGBoost model and predicts efficacy for the given DataFrame.
    Crashes (KeyError) if the DataFrame is missing required features.
    """
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model file not found: {model_path}")

    model = xgb.XGBRegressor()
    model.load_model(model_path)

    expected_cols = model.get_booster().feature_names
    if not expected_cols:
        raise ValueError(f"Model {model_path} does not contain embedded 'feature_names'.")

    # Strictly select expected columns. Will raise KeyError if any are missing.
    X = input_df[expected_cols].apply(pd.to_numeric, errors="coerce")

    # Predict (forces float32 for XGBoost compatibility)
    return model.predict(X.astype(np.float32).values)

In [9]:
from tauso.data.consts import SENSE_START

final_data = final_data[final_data[SENSE_START] != -1]
final_data['cells_per_well'] = 10000


In [17]:
from notebooks.consts import NOTEBOOK_PATH
# predicted_efficiency = predict_aso_efficacy(str(NOTEBOOK_PATH / 'models' / 'UnseenOligoModel' / 'UnseenModel.json'), final_data)
predicted_efficiency = predict_aso_efficacy(str(NOTEBOOK_PATH / 'models' / 'SeenOligoModel' / 'GlobalModel2.json'), final_data)

In [18]:
from tauso.data.consts import *
import pandas as pd
import numpy as np
from scipy.stats import spearmanr

# 1. Add the predictions to your DataFrame
# Ensuring it's a 1D array/list to match the DataFrame length
final_data['predicted_efficiency'] = predicted_efficiency

# 2. Define a helper function to calculate metrics including sample size and modification prevalence
def get_cohort_metrics(group):
    n_samples = len(group)

    # Calculate most common modification and its prevalence
    # value_counts(normalize=True) returns the fraction (0 to 1) of each unique value
    if 'Modification' in group.columns and n_samples > 0:
        mod_counts = group['Modification'].value_counts(normalize=True)
        most_common_mod = mod_counts.index[0] if not mod_counts.empty else np.nan
        mod_prevalence = mod_counts.iloc[0] if not mod_counts.empty else np.nan
    else:
        most_common_mod = np.nan
        mod_prevalence = np.nan

    # Spearman correlation requires at least 2 data points.
    # It also requires variance in the data to not return NaN.
    if n_samples > 1:
        # spearmanr returns a tuple: (correlation, p-value)
        corr, p_val = spearmanr(group[INHIBITION], group['predicted_efficiency'])
        return pd.Series({
            'spearman_correlation': corr,
            'n_samples': n_samples,
            'p_value': p_val,
            'most_common_mod': most_common_mod,
            'mod_prevalence': mod_prevalence
        })
    else:
        return pd.Series({
            'spearman_correlation': np.nan,
            'n_samples': n_samples,
            'p_value': np.nan,
            'most_common_mod': most_common_mod,
            'mod_prevalence': mod_prevalence
        })

# 3. Group by the cross product of CELL_LINE and CANONICAL_GENE
# apply the metrics function (removed include_groups=False to avoid Pandas TypeError)
cohort_metrics = (
    final_data.groupby([CELL_LINE, CANONICAL_GENE])
    .apply(get_cohort_metrics)
    .reset_index()
)

# 4. Filter out any NaNs (cohorts with <2 samples or zero variance)
valid_cohorts = cohort_metrics.dropna(subset=['spearman_correlation']).copy()

# Sort the dataset so the printouts are in order
valid_cohorts = valid_cohorts.sort_values(by='spearman_correlation', ascending=False)

# 5. Helper function to calculate and print summary metrics for a given subset
def print_summary_metrics(df, title):
    if len(df) == 0:
        print(f"--- {title} ---")
        print("No valid cohorts in this subset.\n")
        return

    median_spearman = df['spearman_correlation'].median()
    mean_spearman = df['spearman_correlation'].mean()
    weighted_mean_spearman = np.average(df['spearman_correlation'], weights=df['n_samples'])

    print(f"--- {title} ---")
    print(f"Total Valid Cohorts: {len(df)}")
    print(f"Median Spearman Correlation: {median_spearman:.4f}")
    print(f"Simple Mean Spearman Correlation: {mean_spearman:.4f}")
    print(f"Sample-Size Weighted Mean Correlation: {weighted_mean_spearman:.4f}\n")


# 6. Output Results for ALL Cohorts
print_summary_metrics(valid_cohorts, "ALL COHORTS")

# 7. Output Results for Cohorts strictly > 50 samples
large_cohorts = valid_cohorts[valid_cohorts['n_samples'] > 50]
print_summary_metrics(large_cohorts, "COHORTS > 50 SAMPLES")

# 8. Print Individual Cohort Details
print("--- Individual Cohort Metrics ---")
# Format the prevalence as a readable percentage (e.g., 0.85 -> "85.0%") for the final display
display_df = valid_cohorts.copy()
display_df['mod_prevalence'] = (display_df['mod_prevalence'] * 100).map('{:.1f}%'.format)

print(display_df.to_string(index=False))

--- ALL COHORTS ---
Total Valid Cohorts: 19
Median Spearman Correlation: 0.5175
Simple Mean Spearman Correlation: 0.5070
Sample-Size Weighted Mean Correlation: 0.5685

--- COHORTS > 50 SAMPLES ---
Total Valid Cohorts: 14
Median Spearman Correlation: 0.5676
Simple Mean Spearman Correlation: 0.5199
Sample-Size Weighted Mean Correlation: 0.5690

--- Individual Cohort Metrics ---
          Cell_line Canonical Gene Name  spearman_correlation  n_samples       p_value             most_common_mod mod_prevalence
               A431                SOD1              0.800000          4  2.000000e-01 MOE/5-methylcytosines/deoxy         100.0%
               A431              MALAT1              0.744207       2617  0.000000e+00 cEt/5-methylcytosines/deoxy          99.8%
          SK-MEL-28                IRF4              0.738937       2671  0.000000e+00 cEt/5-methylcytosines/deoxy         100.0%
              HepG2                SOD1              0.687146        273  1.755026e-39 MOE/5-methylcy

/tmp/ipykernel_3331872/3421538292.py:49: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(get_cohort_metrics)


In [19]:
from tauso.data.consts import *
import pandas as pd
import numpy as np
from scipy.stats import spearmanr

# (Assuming final_data already has 'predicted_efficiency' added)

# 1. Helper function for calculating correlation on a pure subset
def get_pure_cohort_metrics(group):
    n_samples = len(group)
    if n_samples > 1:
        # Prevent spearmanr from throwing errors if a subset has zero variance
        if group[INHIBITION].nunique() > 1 and group['predicted_efficiency'].nunique() > 1:
            corr, p_val = spearmanr(group[INHIBITION], group['predicted_efficiency'])
        else:
            corr, p_val = np.nan, np.nan

        return pd.Series({
            'spearman_correlation': corr,
            'n_samples': n_samples,
            'p_value': p_val
        })
    else:
        return pd.Series({
            'spearman_correlation': np.nan,
            'n_samples': n_samples,
            'p_value': np.nan
        })

# 2. THE CRUCIAL CHANGE: Group by Modification AND the Cohort identifiers.
# This forces mixed cohorts (like A431/KRAS) to split into pure sub-groups.
pure_cohort_metrics = (
    final_data.groupby(['Modification', CELL_LINE, CANONICAL_GENE])
    .apply(get_pure_cohort_metrics)
    .reset_index()
)

# 3. Filter out invalid subsets (NaNs or <2 samples)
valid_pure_cohorts = pure_cohort_metrics.dropna(subset=['spearman_correlation']).copy()

# 4. Create the Final Summary function by Chemistry
def summarize_modifications(df):
    summary = []
    # Now we group our pure cohorts strictly by their chemistry
    for mod, group in df.groupby('Modification'):
        n_subcohorts = len(group)
        total_samples = group['n_samples'].sum()
        median_corr = group['spearman_correlation'].median()
        mean_corr = group['spearman_correlation'].mean()
        # Weighted mean prevents N=4 cohorts from skewing the true accuracy of the chemistry
        w_mean_corr = np.average(group['spearman_correlation'], weights=group['n_samples'])

        summary.append({
            'Modification': mod,
            'Pure_Cohorts_Count': n_subcohorts,
            'Total_Samples': total_samples,
            'Median_Corr': median_corr,
            'Mean_Corr': mean_corr,
            'Weighted_Mean_Corr': w_mean_corr
        })

    # Return sorted by Weighted Mean to see the best performing chemistry
    return pd.DataFrame(summary).sort_values('Weighted_Mean_Corr', ascending=False)

# ==========================================
# --- OUTPUT SECTION 1: OVERALL SUMMARIES ---
# ==========================================

print("--- TAUSO Performance by PURE Chemical Modification (ALL SAMPLES) ---")
mod_summary = summarize_modifications(valid_pure_cohorts)

# Format decimals for clean printing
mod_summary['Median_Corr'] = mod_summary['Median_Corr'].map('{:.4f}'.format)
mod_summary['Mean_Corr'] = mod_summary['Mean_Corr'].map('{:.4f}'.format)
mod_summary['Weighted_Mean_Corr'] = mod_summary['Weighted_Mean_Corr'].map('{:.4f}'.format)
print(mod_summary.to_string(index=False))

# Calculate and print summary for >50 samples only
print("\n--- TAUSO Performance by PURE Chemical Modification (> 50 SAMPLES ONLY) ---")
large_pure_cohorts = valid_pure_cohorts[valid_pure_cohorts['n_samples'] > 50].copy()

if not large_pure_cohorts.empty:
    large_mod_summary = summarize_modifications(large_pure_cohorts)
    large_mod_summary['Median_Corr'] = large_mod_summary['Median_Corr'].map('{:.4f}'.format)
    large_mod_summary['Mean_Corr'] = large_mod_summary['Mean_Corr'].map('{:.4f}'.format)
    large_mod_summary['Weighted_Mean_Corr'] = large_mod_summary['Weighted_Mean_Corr'].map('{:.4f}'.format)
    print(large_mod_summary.to_string(index=False))
else:
    print("No pure cohorts with > 50 samples found.")


# ==========================================
# --- OUTPUT SECTION 2: INDIVIDUAL COHORTS ---
# ==========================================
display_cols = [CELL_LINE, CANONICAL_GENE, 'Modification', 'spearman_correlation', 'n_samples', 'p_value']

print("\n\n--- Individual PURE Cohort Breakdown (ALL SAMPLES) ---")
sorted_all = valid_pure_cohorts.sort_values(by='spearman_correlation', ascending=False)
print(sorted_all[display_cols].to_string(index=False))

print("\n\n--- Individual PURE Cohort Breakdown (> 50 SAMPLES ONLY) ---")
sorted_large = large_pure_cohorts.sort_values(by='spearman_correlation', ascending=False)
print(sorted_large[display_cols].to_string(index=False))


# ==========================================
# --- OUTPUT SECTION 3: DEEP DIVE ---
# ==========================================
print("\n\n--- Deep Dive: How A431/KRAS was split ---")
# Let's peek specifically at how A431/KRAS performed across its two chemistries
a431_kras = valid_pure_cohorts[
    (valid_pure_cohorts[CELL_LINE] == 'A431') &
    (valid_pure_cohorts[CANONICAL_GENE] == 'KRAS')
]
print(a431_kras[display_cols].to_string(index=False))

--- TAUSO Performance by PURE Chemical Modification (ALL SAMPLES) ---
                   Modification  Pure_Cohorts_Count  Total_Samples Median_Corr Mean_Corr Weighted_Mean_Corr
    cEt/5-methylcytosines/deoxy                  12        12615.0      0.5582    0.5261             0.6145
(S)-cEt/5-methylcytosines/deoxy                   1          235.0      0.5869    0.5869             0.5869
    MOE/5-methylcytosines/deoxy                   5         3046.0      0.6871    0.6847             0.5601
                      LNA/deoxy                   3         2256.0      0.3269    0.2452             0.3225

--- TAUSO Performance by PURE Chemical Modification (> 50 SAMPLES ONLY) ---
                   Modification  Pure_Cohorts_Count  Total_Samples Median_Corr Mean_Corr Weighted_Mean_Corr
    cEt/5-methylcytosines/deoxy                   8        12538.0      0.6485    0.5960             0.6155
(S)-cEt/5-methylcytosines/deoxy                   1          235.0      0.5869    0.5869         

/tmp/ipykernel_3331872/552842817.py:34: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(get_pure_cohort_metrics)


In [13]:
def calculate_spearman(group):
    # Spearman requires at least 2 data points with variance
    if len(group) > 1:
        corr, p_value = spearmanr(group[INHIBITION], group['predicted_efficiency'])
        return corr
    return np.nan

# ==========================================
# CELL 4: Subset Evaluation by Modification Type
# ==========================================

# 1. Define the exact modification strings you want to isolate
target_mods = [
    'cEt/5-methylcytosines/deoxy',
    'MOE/5-methylcytosines/deoxy',
    'LNA/deoxy'
]

print("--- Median Spearman Correlation by Modification Type ---\n")

# 2. Iterate through each modification type individually
for mod in target_mods:
    # Filter results for just this specific modification
    mod_df = final_data[final_data['Modification'] == mod].copy()

    # Drop rows missing predictions or true values
    eval_mod_df = mod_df.dropna(subset=[INHIBITION, 'predicted_efficiency'])

    subset_size = len(eval_mod_df)

    # Check if we have any data before proceeding
    if subset_size == 0:
        print(f"[{mod}] (N=0 rows)")
        print(f"    -> Cannot calculate: No data found for this modification.\n")
        continue

    # 3. Group and calculate Spearman (re-using your calculate_spearman function)
    grouped_corrs = eval_mod_df.groupby(
        [CANONICAL_GENE, CELL_LINE]
    ).apply(calculate_spearman).reset_index(name='spearman_corr')

    # Drop groups that couldn't be calculated (e.g., < 2 items or no variance)
    grouped_corrs = grouped_corrs.dropna(subset=['spearman_corr'])

    # 4. Calculate and report the final median for this mod
    if not grouped_corrs.empty:
        median_val = grouped_corrs['spearman_corr'].median()
        num_groups = len(grouped_corrs)
        print(f"[{mod}] (N={subset_size} rows)")
        print(f"    -> Median Spearman: {median_val:.4f} (Calculated across {num_groups} Gene/Cell Line groups)\n")
    else:
        print(f"[{mod}] (N={subset_size} rows)")
        print(f"    -> Cannot calculate Spearman: Not enough variance within the individual groups.\n")

--- Median Spearman Correlation by Modification Type ---

[cEt/5-methylcytosines/deoxy] (N=12615 rows)
    -> Median Spearman: 0.5218 (Calculated across 12 Gene/Cell Line groups)

[MOE/5-methylcytosines/deoxy] (N=3046 rows)
    -> Median Spearman: 0.2379 (Calculated across 5 Gene/Cell Line groups)

[LNA/deoxy] (N=2256 rows)
    -> Median Spearman: 0.3145 (Calculated across 3 Gene/Cell Line groups)



/tmp/ipykernel_3331872/3403295746.py:40: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ).apply(calculate_spearman).reset_index(name='spearman_corr')
/tmp/ipykernel_3331872/3403295746.py:40: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ).apply(calculate_spearman).reset_index(name='spearman_corr')
/tmp/ipykernel_3331872/3403295746.py:40: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This beha